# Topic 19.1 - Data Gathering

## From public sources to analysis-ready data

Data analysis does not start when a chart appears. It starts when an analyst decides what evidence is needed, finds a credible source, retrieves the data correctly, records where it came from, and stores it in a form that can be checked and reused.

This lesson treats data gathering as a complete stage of the analysis process. We will work with real public data from Our World in Data, the World Bank Indicators API, GitHub's REST API, and the Gapminder country-development dataset. We will practise four routes:

1. downloading a CSV from a URL;
2. requesting and parsing JSON from an API;
3. loading real observations into a relational SQLite database;
4. preserving nested JSON documents in a non-relational key-value/document workflow.

The files used by the examples are cached in `data_analysis_process/data/` so the notebook remains useful when a student is offline. Each section also shows how to refresh or request the source again.

## Learning objectives

By the end of this lesson, you should be able to:

- explain the difference between primary and secondary data and between structured, semi-structured, and unstructured data;
- evaluate a dataset's publisher, purpose, grain, units, time coverage, licence, and limitations before using it;
- download a public CSV or JSON file with Python and retain provenance information;
- construct an API request with parameters, inspect status and metadata, handle missing values, and flatten nested JSON;
- describe tables, rows, columns, primary keys, foreign keys, joins, and constraints in a relational database;
- use SQLite and SQL to store and query a real dataset;
- explain document and key-value database ideas and practise them with real GitHub API documents;
- compare collection methods and choose a method that fits a question and an access pattern;
- create a small, reproducible data-gathering manifest.

## The data-gathering workflow

| Stage | Main question | Evidence to keep |
|---|---|---|
| Define | What do we need to know? | Question, variables, population, time period |
| Evaluate | Can we trust and use this source? | Publisher, documentation, licence, limitations |
| Retrieve | How do we obtain the records? | URL, endpoint, parameters, status, timestamp |
| Preserve | Can another person reproduce the input? | Raw file, raw API response, checksum, manifest |
| Inspect | What does one record mean? | Shape, types, missingness, grain, data dictionary |
| Store | What structure supports future work? | CSV, SQL tables, documents, keys, constraints |
| Handoff | Can analysis begin responsibly? | Clean access layer and written assumptions |

Keep the raw source separate from cleaned or derived data. A raw file is evidence of what was received; a cleaned table is an interpretation that should be reproducible from the raw input.

## 1. What does data gathering mean?

Data gathering is the deliberate process of locating, accessing, collecting, documenting, and preserving observations that can help answer a question. It can happen through a download, an API request, a database query, a survey, a sensor, a form, or another approved collection method.

### Important distinctions

| Distinction | Meaning | Example |
|---|---|---|
| Primary data | Collected directly for the current project | A school survey designed by your team |
| Secondary data | Collected by somebody else and reused | World Bank population estimates |
| Structured | Fixed rows and columns | A CSV table |
| Semi-structured | Nested or flexible fields with tags/keys | JSON from an API |
| Unstructured | Does not follow a table schema | Interviews, images, free-form documents |
| Batch | Gathered periodically or in a one-time download | Annual census CSV |
| Streaming | Arrives continuously or in small events | Sensor readings or transaction events |

The collection method affects what claims are possible. A convenient dataset may not represent the population you care about, and an API response may change after you have used it. Understanding those limits is part of analysis.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import mimetypes
import shelve
import sqlite3
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import pandas as pd

# Works when Jupyter starts in the repository root or this lesson folder.
if (Path.cwd() / "data_analysis_process" / "data").exists():
    LESSON_DIR = Path.cwd() / "data_analysis_process"
else:
    LESSON_DIR = Path.cwd()
DATA_DIR = LESSON_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
print("Lesson directory:", LESSON_DIR)
print("Data directory:", DATA_DIR)

Lesson directory: /home/johnny/Projects/ITFY-Ghana/data_analysis_process
Data directory: /home/johnny/Projects/ITFY-Ghana/data_analysis_process/data


### Try It 0 - write a source plan

Before gathering any data, write a short plan for a question such as:

> How has life expectancy changed in Ghana compared with two other countries?

State the unit of analysis, the time period, at least two variables, the population represented, and one limitation you expect. This prevents the source from deciding the question for you.

In [ ]:
# YOUR NOTES HERE
question = ""
unit_of_analysis = ""
time_period = ""
variables = []
expected_limitation = ""

## 2. Gathering data from a URL

A URL is an address. It may point to a web page, an API endpoint, or the actual bytes of a file. These are different things:

| URL target | What you receive | Typical next step |
|---|---|---|
| Web page | HTML intended for people | Read the page or use a documented export |
| CSV file | Delimited text such as country,year,population | Save it, then use `pd.read_csv()` |
| JSON file | Objects, arrays, and nested values | Parse with `json` or `pd.json_normalize()` |
| ZIP archive | One or more compressed files | Download, inspect, then extract carefully |

The visible download button on a website is often backed by a direct file URL. Prefer a documented download URL or API endpoint over scraping a page designed only for human viewing.

### URL anatomy

    https://api.worldbank.org/v2/country/GHA/indicator/SP.POP.TOTL?date=2020:2022&format=json
    scheme       path                                  query parameters

The query parameters after `?` change the request. An ampersand separates parameters. A colon can have meaning to the API, so use `urllib.parse.urlencode()` when building a URL in Python instead of manually concatenating untrusted values.

In [ ]:
GAPMINDER_URL = "https://raw.githubusercontent.com/plotly/datasets/master/gapminderDataFiveYear.csv"
OWID_LIFE_EXPECTANCY_URL = "https://ourworldindata.org/grapher/life-expectancy.csv"


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def download_file(url, destination, force=False, timeout=30):
    """Download a file once, retain it locally, and return basic provenance."""
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and not force:
        print(f"Using cached file: {destination}")
    else:
        request = Request(url, headers={"User-Agent": "ITFY-Ghana-data-gathering-lesson/1.0"})
        with urlopen(request, timeout=timeout) as response:
            status = getattr(response, "status", None)
            content = response.read()
        if status is not None and status != 200:
            raise RuntimeError(f"Download failed with HTTP status {status}")
        destination.write_bytes(content)
        print(f"Downloaded {len(content):,} bytes to {destination}")
    return {
        "source_url": url,
        "local_path": str(destination),
        "retrieved_utc": datetime.now(timezone.utc).isoformat(),
        "sha256": sha256_file(destination),
        "media_type": mimetypes.guess_type(destination.name)[0],
    }

In [ ]:
# These files are already included in the repository. force=False demonstrates
# the same URL workflow without downloading the same bytes again.
gapminder_provenance = download_file(GAPMINDER_URL, DATA_DIR / "gapminder_five_year.csv")
life_expectancy_provenance = download_file(OWID_LIFE_EXPECTANCY_URL, DATA_DIR / "owid_life_expectancy.csv")
pd.DataFrame([gapminder_provenance, life_expectancy_provenance])

### What makes a download reproducible?

Saving a file is only part of the job. A future analyst should be able to identify:

1. the exact source URL;
2. the date and time it was retrieved;
3. the local filename used by the project;
4. the file format and encoding;
5. the publisher and licence;
6. the checksum, when an unchanged byte-for-byte copy matters.

The SHA-256 value is a fingerprint of the downloaded bytes. It does not prove that a dataset is correct, but it helps detect an unexpected change in a file that claims to be the same version.

In [ ]:
gapminder = pd.read_csv(DATA_DIR / "gapminder_five_year.csv")
life_expectancy = pd.read_csv(DATA_DIR / "owid_life_expectancy.csv")
print("Gapminder shape:", gapminder.shape)
print("Gapminder columns:", list(gapminder.columns))
print("\nOur World in Data shape:", life_expectancy.shape)
print("Our World in Data columns:", list(life_expectancy.columns))
gapminder.head()

### Try It 1 - download and inspect another real CSV

Use the function above to download the Our World in Data life expectancy CSV into a new local filename, or choose another CSV from a trustworthy source. Then:

- print its shape and column names;
- display the first five rows;
- identify what one row represents;
- record the URL and a short note about the publisher.

Do not begin by plotting. First establish the grain of the data: one place in one year, one person, one transaction, or another unit.

In [ ]:
# YOUR CODE HERE
# provenance = download_file(OWID_LIFE_EXPECTANCY_URL, DATA_DIR / "life_expectancy_copy.csv")
# downloaded = pd.read_csv(DATA_DIR / "life_expectancy_copy.csv")
# print(downloaded.shape)
# downloaded.head()

## 3. Gathering data from an API

An API is a program-friendly doorway into a service. Instead of downloading one large file, we ask for a particular resource and may specify filters such as country, year, indicator, page, or format.

### The request-response cycle

1. The client constructs a URL and sends an HTTP request.
2. The server returns a status code, headers, and a response body.
3. The client checks the status and content type.
4. The client parses JSON, XML, CSV, or another documented format.
5. The client records the request parameters and source metadata.

The World Bank Indicators API is a useful teaching example because it provides public development indicators, uses documented indicator codes, supports JSON, and does not require an API key for ordinary requests. Its documentation explains the V2 URL structure and query parameters.

We will use these real indicators for Ghana, Nigeria, Kenya, and South Africa:

| Indicator | Code | Meaning |
|---|---|---|
| Population, total | `SP.POP.TOTL` | Total population |
| GDP per capita, current US$ | `NY.GDP.PCAP.CD` | GDP divided by population in current US dollars |
| Life expectancy at birth | `SP.DYN.LE00.IN` | Expected years at birth |

The source response is an array: the first item is pagination and source metadata; the second item is the list of observations. This is a common API pattern and is different from a simple CSV table.

In [ ]:
WORLD_BANK_DOCS = "https://datahelpdesk.worldbank.org/knowledgebase/articles/889392"
WORLD_BANK_API = "https://api.worldbank.org/v2/country/GHA;NGA;KEN;ZAF/indicator/SP.POP.TOTL"


def fetch_json(url, params=None, timeout=30):
    """Make a documented GET request and return (parsed_json, final_url)."""
    params = params or {}
    query = urlencode(params)
    request_url = f"{url}?{query}" if query else url
    request = Request(request_url, headers={
        "Accept": "application/json",
        "User-Agent": "ITFY-Ghana-data-gathering-lesson/1.0",
    })
    try:
        with urlopen(request, timeout=timeout) as response:
            if response.status != 200:
                raise RuntimeError(f"HTTP {response.status} from {request_url}")
            return json.load(response), request_url
    except Exception as error:
        raise RuntimeError(f"Could not retrieve {request_url}: {error}") from error


def read_world_bank_records(path):
    """Read a saved World Bank V2 response and return its observations."""
    payload = json.loads(Path(path).read_text(encoding="utf-8"))
    if isinstance(payload, list) and payload and isinstance(payload[0], dict) and "message" in payload[0]:
        raise ValueError(f"World Bank returned an API error: {payload[0]['message']}")
    if not isinstance(payload, list) or len(payload) < 2:
        raise ValueError("Expected [metadata, records] from the World Bank API")
    return pd.json_normalize(payload[1]), payload[0]

In [ ]:
# Use saved API responses for a stable lesson. The same parser also works with
# a fresh response returned by fetch_json().
api_files = {
    "population": DATA_DIR / "world_bank_population_api_raw.json",
    "gdp_per_capita": DATA_DIR / "world_bank_gdp_per_capita_api_raw.json",
    "life_expectancy": DATA_DIR / "world_bank_life_expectancy_api_raw.json",
}
api_frames = []
api_metadata = {}
for indicator_name, path in api_files.items():
    frame, metadata = read_world_bank_records(path)
    frame = frame.rename(columns={
        "country.value": "country",
        "countryiso3code": "country_code",
        "date": "year",
        "value": indicator_name,
    })
    frame["year"] = frame["year"].astype(int)
    api_frames.append(frame[["country", "country_code", "year", indicator_name]])
    api_metadata[indicator_name] = metadata
world_bank_tidy = api_frames[0]
for frame in api_frames[1:]:
    world_bank_tidy = world_bank_tidy.merge(frame, on=["country", "country_code", "year"], how="outer")
world_bank_tidy.sort_values(["year", "country"]).head(12)

In [ ]:
print("API metadata for the population response:")
print(api_metadata["population"])
print("\nTidy API data:")
world_bank_tidy.info()
world_bank_tidy.tail()

### API details students must learn

- Parameters narrow a response with documented keys such as `date`, `format`, `page`, or `per_page`.
- Pagination matters. Inspect `page`, `pages`, and `total` before assuming you received everything.
- Status codes distinguish successful, authentication, rate-limit, missing-resource, and server errors.
- Headers such as `Accept`, `User-Agent`, `ETag`, and `Last-Modified` affect representation and caching.
- Authentication may use an API key, OAuth token, or signed request. Never place a secret directly in a shared notebook.
- Rate limits require caching, smaller requests, and respect for the provider's terms.
- A JSON `null` is a missing observation, not the number zero.
- Technical accessibility does not remove licence, privacy, or attribution obligations.

The file `world_bank_development_api_raw.json` is intentionally kept as an example of a failed request. Its error message shows why checking the response matters: a request can return valid JSON while still describing an API error.

In [ ]:
failed_request = DATA_DIR / "world_bank_development_api_raw.json"
failed_payload = json.loads(failed_request.read_text(encoding="utf-8"))
print("Saved failed response:", failed_payload)
try:
    read_world_bank_records(failed_request)
except ValueError as error:
    print("Parser caught the problem:", error)

### Try It 2 - request and parse an API response

Use the World Bank API to retrieve total population for Ghana and Nigeria from 2010 to 2022. Use `fetch_json()` and `pd.json_normalize()`.

Your final table should have one row per country-year and these columns:

    country | country_code | year | population

Then answer with code: in which year did Ghana's population first exceed 30 million in this response?

In [ ]:
world_bank_base = "https://api.worldbank.org/v2/country/GHA;NGA/indicator/SP.POP.TOTL"
params = {"date": "2010:2022", "format": "json", "per_page": 1000}

# YOUR CODE HERE
# Fetch the response, parse its records, and create a tidy DataFrame with
# country, year, and population columns. Then compare Ghana and Nigeria.

## 4. Gathering data from a relational database

A relational database stores data in tables. A table has rows and columns, and relationships are represented with keys.

| Database idea | Example in this lesson |
|---|---|
| Table | `countries` or `development_observations` |
| Row | One country, or one country in one year |
| Column | `continent`, `year`, or `life_expectancy` |
| Primary key | A value that uniquely identifies a row |
| Foreign key | A value that points to a row in another table |
| Join | Combine related tables using matching keys |
| Constraint | A rule such as `NOT NULL`, `UNIQUE`, or `FOREIGN KEY` |

Relational design is valuable when the data has stable relationships, consistency rules, and many questions that combine entities. The database engine can filter and aggregate data without sending every row to Python first.

We will build a small SQLite database from the real Gapminder country-development dataset already used in the investigation lesson. SQLite is relational, serverless, and included with Python's standard library. The same SQL ideas transfer to PostgreSQL, MySQL, MariaDB, SQL Server, and other relational systems.

In [ ]:
development = gapminder.rename(columns={
    "country": "country_name",
    "pop": "population",
    "lifeExp": "life_expectancy",
    "gdpPercap": "gdp_per_capita",
}).copy()
country_lookup = development[["country_name", "continent"]].drop_duplicates().sort_values("country_name").reset_index(drop=True)
country_lookup["country_id"] = country_lookup.index + 1
development = development.merge(country_lookup, on=["country_name", "continent"], how="left")
development["year"] = development["year"].astype(int)

db_path = DATA_DIR / "development_relational.sqlite"
conn = sqlite3.connect(db_path)
conn.execute("PRAGMA foreign_keys = ON")
conn.executescript("""
DROP TABLE IF EXISTS development_observations;
DROP TABLE IF EXISTS countries;
CREATE TABLE countries (
    country_id INTEGER PRIMARY KEY,
    country_name TEXT NOT NULL UNIQUE,
    continent TEXT NOT NULL
);
CREATE TABLE development_observations (
    country_id INTEGER NOT NULL,
    year INTEGER NOT NULL,
    population INTEGER NOT NULL,
    life_expectancy REAL NOT NULL,
    gdp_per_capita REAL NOT NULL,
    PRIMARY KEY (country_id, year),
    FOREIGN KEY (country_id) REFERENCES countries(country_id)
);
""")
conn.executemany(
    "INSERT INTO countries (country_id, country_name, continent) VALUES (?, ?, ?)",
    country_lookup[["country_id", "country_name", "continent"]].itertuples(index=False, name=None),
)
conn.executemany(
    "INSERT INTO development_observations (country_id, year, population, life_expectancy, gdp_per_capita) VALUES (?, ?, ?, ?, ?)",
    development[["country_id", "year", "population", "life_expectancy", "gdp_per_capita"]].itertuples(index=False, name=None),
)
conn.commit()
print(f"Created {db_path}")
print("Countries:", len(country_lookup))
print("Observations:", len(development))

In [ ]:
print(pd.read_sql_query("SELECT name, sql FROM sqlite_master WHERE type = 'table' ORDER BY name", conn).to_string(index=False))
continent_year_summary = pd.read_sql_query(
    """
    SELECT c.continent, o.year, COUNT(*) AS countries,
           ROUND(AVG(o.life_expectancy), 2) AS average_life_expectancy,
           ROUND(AVG(o.gdp_per_capita), 2) AS average_gdp_per_capita
    FROM development_observations AS o
    JOIN countries AS c ON c.country_id = o.country_id
    GROUP BY c.continent, o.year
    ORDER BY o.year, c.continent
    """,
    conn,
)
continent_year_summary.head(10)

### Why the join matters

The observation table stores measurements and a `country_id`. The country table stores descriptive attributes such as the country name and continent. This avoids repeating the continent name in every observation row and gives us one place to update country information.

The SQL query below uses a parameter placeholder (`?`) instead of inserting a value into a string. Parameterized queries are safer and more portable because the database driver handles the value separately from the SQL statement.

In [ ]:
country_query = """
SELECT c.country_name, c.continent, o.year, o.population, o.life_expectancy, o.gdp_per_capita
FROM development_observations AS o
JOIN countries AS c ON c.country_id = o.country_id
WHERE c.country_name = ? AND o.year >= ?
ORDER BY o.year
"""
ghana_from_sql = pd.read_sql_query(country_query, conn, params=("Ghana", 2000))
ghana_from_sql

In [ ]:
top_2007 = pd.read_sql_query(
    """
    SELECT c.country_name, c.continent, o.life_expectancy, o.gdp_per_capita
    FROM development_observations AS o
    JOIN countries AS c ON c.country_id = o.country_id
    WHERE o.year = 2007
    ORDER BY o.gdp_per_capita DESC
    LIMIT 10
    """,
    conn,
)
top_2007

### Production database connections

SQLite is ideal for a self-contained lesson, prototypes, mobile apps, and local analysis. In a team system, the same data might live in PostgreSQL or MySQL. A typical production workflow is:

1. obtain credentials from a secret manager or environment variables;
2. connect through a driver such as `psycopg` or `mysql-connector-python`;
3. use a connection pool and parameterized SQL;
4. select only the columns and rows required;
5. close the connection or return it to the pool.

Never commit database passwords, connection strings, or API keys to a notebook or Git repository.

### Try It 3 - query the relational database

Write a SQL query that returns the 2007 country with the highest life expectancy. Include the country name, continent, life expectancy, and GDP per capita. Then change the query to return the top five countries instead of one.

In [ ]:
# YOUR CODE HERE
# Query the SQLite database for the 2007 country with the highest life
# expectancy. Return country, continent, life expectancy, and GDP per capita.

## 5. Gathering data from non-relational databases

"Non-relational" is a family of designs, not one single database. Common categories include:

| Type | Natural shape | Examples |
|---|---|---|
| Document | JSON-like documents, nested objects, arrays | MongoDB, CouchDB, Firestore |
| Key-value | A key maps directly to a value | Redis, DynamoDB key-value access, `shelve` |
| Wide-column | Rows can have different columns and scale across partitions | Cassandra, HBase |
| Graph | Nodes and relationships | Neo4j, Amazon Neptune |

Relational databases usually encourage a predefined schema and separate related tables. A document database stores a document with field-value pairs and can embed arrays or nested documents. The flexible shape can be useful when records naturally vary, but it does not remove the need for thoughtful modelling, validation, indexes, or documentation.

For a real example, we use public repository documents returned by GitHub's REST API. The saved file contains fields such as `full_name`, `owner`, `topics`, `license`, `stargazers_count`, and timestamps. Notice that `owner` is itself an object and `topics` is an array: this is a natural document shape.

In [ ]:
github_documents = json.loads((DATA_DIR / "github_plotly_repositories.json").read_text(encoding="utf-8"))
print("Number of repository documents:", len(github_documents))
print("First document keys:", list(github_documents[0].keys()))
print("Nested owner document:", github_documents[0]["owner"])
print("Topics array:", github_documents[0].get("topics"))

In [ ]:
python_repositories = [document for document in github_documents if document.get("language") == "Python"]
python_repositories = sorted(python_repositories, key=lambda document: document.get("stargazers_count", 0), reverse=True)
pd.DataFrame([
    {
        "repository": document["full_name"],
        "owner": document["owner"]["login"],
        "stars": document["stargazers_count"],
        "open_issues": document["open_issues_count"],
        "topics": ", ".join(document.get("topics") or []),
    }
    for document in python_repositories[:10]
])

### A local key-value database with real JSON documents

Python's standard-library `shelve` module provides a small persistent key-value store. It is useful here because students can run the example without installing a database server. The value is still a complete GitHub document, so the access pattern is:

    key:   plotly/datasets
    value: {"id": ..., "owner": {...}, "topics": [...], ...}

This is not a replacement for a production database. It is a bridge between reading JSON in memory and understanding how a key-value or document service stores records.

In [ ]:
with shelve.open(str(DATA_DIR / "github_repositories_kv")) as repository_store:
    for document in github_documents:
        repository_store[document["full_name"]] = document
    print("Stored documents:", len(repository_store))
with shelve.open(str(DATA_DIR / "github_repositories_kv")) as repository_store:
    first_key = sorted(repository_store.keys())[0]
    one_document = repository_store[first_key]
    print("Retrieved by key:", first_key)
    print("Owner login:", one_document["owner"].get("login"))
    print("Stars:", one_document.get("stargazers_count"))

### Optional MongoDB connection

MongoDB is a widely used document database. The following code is intentionally optional: it requires a running MongoDB deployment and a connection URI stored in the `MONGODB_URI` environment variable. The data itself is real and local; the remote database is only needed to practise insertion and querying.

The MongoDB equivalent of a SQL table is a collection. A document can be inserted with `insert_one()` or `insert_many()`, filtered with `find()`, and aggregated with an aggregation pipeline. MongoDB documentation also explains when to embed related data and when to use references.

In [ ]:
# Optional: run only after installing pymongo and setting MONGODB_URI.
import os
if "MONGODB_URI" not in os.environ:
    print("Set MONGODB_URI before connecting to MongoDB.")
else:
    try:
        from pymongo import MongoClient
        client = MongoClient(os.environ["MONGODB_URI"], serverSelectionTimeoutMS=5000)
        collection = client["itfy_data_gathering"]["plotly_repositories"]
        collection.delete_many({})
        collection.insert_many(github_documents)
        result = collection.find(
            {"language": "Python", "stargazers_count": {"$gte": 0}},
            {"_id": 0, "full_name": 1, "stargazers_count": 1, "owner.login": 1},
        ).sort("stargazers_count", -1).limit(5)
        list(result)
    except ImportError:
        print("Install pymongo in your environment to run this optional example.")
    except Exception as error:
        print("MongoDB connection was not available:", error)

### Relational and document thinking side by side

| Question | SQLite | MongoDB-style document query |
|---|---|---|
| Python repositories | `WHERE language = 'Python'` | `find({'language': 'Python'})` |
| At least 100 stars | `WHERE stargazers_count >= 100` | `find({'stargazers_count': {'$gte': 100}})` |
| Sort by stars | `ORDER BY stargazers_count DESC` | `.sort('stargazers_count', -1)` |
| Nested owner login | joined or flattened column | `owner.login` |

The database should match the access patterns and consistency requirements of the application. Choosing NoSQL simply because it is popular is not a data model.

### Try It 4 - work with document data

Load the saved GitHub repository documents. Keep public repositories with a non-null language, sort them by stars, display the top five, and store them in `shelve` using `full_name` as the key. Then retrieve one document by key and print its nested owner login.

In [ ]:
# YOUR CODE HERE
# 1. Load github_plotly_repositories.json.
# 2. Keep public repositories with a non-null language.
# 3. Sort by stargazers_count and display the top five names and stars.
# 4. Store the documents in a shelve key-value store using full_name as key.

## 6. Provenance, quality, ethics, and safety

Data gathering is a technical and social responsibility. Before analysis, ask:

### Source quality

- Who published the data and why?
- Is there a primary source or only a copied version?
- Is the dataset current enough for the question?
- Is the unit clearly defined?
- Does one row represent what we think it represents?
- Are missing values, revisions, estimates, and forecasts documented?

### Provenance

Keep a small manifest alongside the data. A useful manifest records the source URL, retrieval time, local filename, format, licence or attribution, parameters, and checksum. For an API, also record the endpoint, indicator codes, date range, page size, and whether the response was complete.

### Ethics and privacy

Publicly accessible does not automatically mean ethically unrestricted. Avoid collecting personal data that is not necessary. Read terms of use and licences. Remove or protect names, phone numbers, email addresses, exact locations, health information, and other identifiers when a project does not need them. Ask whether a vulnerable person could be harmed by publication or re-identification.

### Data protection in code

Use environment variables or a secret manager for credentials. Avoid printing access tokens in outputs. Do not commit raw confidential files. Keep raw data immutable and create a cleaned copy for transformations so the provenance chain remains visible.

In [ ]:
source_manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "sources": [
        {
            "name": "Gapminder five-year indicators",
            "url": GAPMINDER_URL,
            "local_file": "gapminder_five_year.csv",
            "format": "CSV",
            "notes": "Country-year observations used for the SQLite demonstration.",
        },
        {
            "name": "Our World in Data life expectancy",
            "url": OWID_LIFE_EXPECTANCY_URL,
            "local_file": "owid_life_expectancy.csv",
            "format": "CSV",
            "notes": "One place-year observation with a life expectancy value.",
        },
        {
            "name": "World Bank Indicators API",
            "url": WORLD_BANK_API,
            "local_files": ["world_bank_population_api_raw.json", "world_bank_gdp_per_capita_api_raw.json", "world_bank_life_expectancy_api_raw.json"],
            "format": "JSON",
            "parameters": {"countries": "GHA;NGA;KEN;ZAF", "years": "2000:2022"},
            "notes": "Three indicator requests saved as raw responses.",
        },
        {
            "name": "GitHub REST API - Plotly repositories",
            "url": "https://api.github.com/orgs/plotly/repos?per_page=30&sort=stars&direction=desc",
            "local_file": "github_plotly_repositories.json",
            "format": "JSON documents",
            "notes": "Repository documents used for the document/key-value example.",
        },
    ],
}
manifest_path = DATA_DIR / "topic19_data_gathering_manifest.json"
manifest_path.write_text(json.dumps(source_manifest, indent=2), encoding="utf-8")
print(f"Wrote provenance manifest to {manifest_path}")

## 7. Choosing a gathering method

| Situation | Good starting point | Why |
|---|---|---|
| A stable published table is available | Direct CSV or ZIP URL | Easy to archive and reproduce |
| You need a current slice or selected indicators | Documented API | Request only the data needed |
| Many users need consistent relationships and transactions | Relational database | Keys, constraints, joins, and SQL |
| Records vary naturally or contain nested objects | Document database | Flexible documents and embedded arrays |
| Very fast lookup by one known key | Key-value store | Simple access path and low overhead |
| Relationships are the main object of the question | Graph database | Nodes and edges make paths explicit |

The strongest projects often use more than one method: an API may supply raw records, a relational database may store validated facts, and a document store may preserve the original nested response for application use.

## 8. Mini capstone - build a source-to-storage pipeline

Choose a question that compares Ghana with at least one other country. Complete all of these stages:

1. Plan: write the question, target population, time period, variables, and acceptable evidence.
2. Find: identify a trustworthy URL, API, or database source and read its documentation.
3. Gather: download or request the data; keep the raw response unchanged.
4. Record: create a provenance entry with URL, parameters, retrieval time, licence, and checksum where appropriate.
5. Inspect: report the shape, grain, data types, missingness, and unusual values.
6. Store: save the data in a small SQLite table and, separately, preserve one nested JSON record in a document/key-value form.
7. Explain: describe why each storage method fits the data and what could go wrong.

Your final notebook should contain code outputs and a short paragraph after every stage. A chart is welcome, but the goal of this lesson is a reliable gathering pipeline rather than a polished visualisation.

## Key takeaways

- Data gathering begins with a question and source evaluation, not with `read_csv()`.
- A URL may point to a web page, a file, or an API response; identify which one you have.
- APIs require careful handling of parameters, pagination, status codes, missing values, and credentials.
- Relational databases organise stable relationships with tables, keys, constraints, and SQL.
- Non-relational databases include document, key-value, wide-column, and graph designs.
- Real work is reproducible when raw data, metadata, provenance, and assumptions are kept together.
- A source can be accessible and still be unsuitable, misunderstood, unlicensed, or unsafe to publish.

## Further reading and the course path

- [World Bank Indicators API documentation](https://datahelpdesk.worldbank.org/knowledgebase/articles/889392)
- [World Bank API basic call structures](https://datahelpdesk.worldbank.org/knowledgebase/articles/898581-api-basic-call-structures)
- [GitHub REST API repository endpoints](https://docs.github.com/en/rest/repos)
- [SQLite SQL language reference](https://sqlite.org/lang.html)
- [SQLite SELECT documentation](https://www.sqlite.org/lang_select.html)
- [MongoDB introduction and document model](https://www.mongodb.com/docs/manual/introduction/index.html)
- [MongoDB data modelling](https://www.mongodb.com/docs/v5.1/core/data-modeling-introduction/)
- [Our World in Data grapher](https://ourworldindata.org/grapher)
- [Common data sources appendix](topic19b_appendix_common_data_sources.ipynb)
- [Reading dataset documentation appendix](topic19c_appendix_reading_dataset_documentation.ipynb)
- [Data gathering and understanding before analysis](topic19a_data_gathering_and_understanding.ipynb)
- [Full global development analysis](topic19_global_development_analysis.ipynb)
- [Topic 19 data gathering assignment](assignments/topic19_assignment_data_gathering.ipynb)